In [1]:
!pip install -q gradio ultralytics tensorflow opencv-python-headless numpy

print('✅ Dépendances installées.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.7 MB/s eta 0:00:00
✅ Dépendances installées.


In [2]:
import tempfile
import os
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional

import cv2
import gradio as gr
import numpy as np
import tensorflow as tf
from ultralytics import YOLO

# ── Chemin du modèle CNN (sera mis à jour après upload à la cellule 3) ────────
CNN_MODEL_PATH  = "mon_modele_v2.keras"   # modifiable
YOLO_MODEL_PATH = "yolov8n.pt"            # téléchargé automatiquement par YOLO

# ── Classes CNN (doit correspondre à l'ordre d'entraînement) ─────────────────
CLASSES = ["bus", "camion", "moto", "voiture"]

# ── IDs YOLO COCO correspondant aux véhicules ─────────────────────────────────
#   2 = car   3 = motorcycle   5 = bus   7 = truck
VEHICULE_IDS = {2, 3, 5, 7}

# ── Couleurs BGR par classe ───────────────────────────────────────────────────
COULEURS: dict = {
    "voiture": (0,   200,   0),
    "moto":    (0,   150, 255),
    "bus":     (255, 100,   0),
    "camion":  (200,   0, 200),
}
COULEUR_ALERTE = (0, 0, 255)   # rouge → dépassement de vitesse

# ── Paramètres CNN ────────────────────────────────────────────────────────────
CNN_INPUT_SIZE    = (64, 64)
CNN_MIN_CROP_SIZE = 20          # px minimum pour un crop valide

# ── Paramètres tracker ────────────────────────────────────────────────────────
CONF_SEUIL              = 0.5   # confiance YOLO minimale
VITESSE_WINDOW          = 5     # frames fenêtre glissante
VITESSE_MAX             = 200   # km/h plafond
SNAPSHOT_INTERVALLE_SEC = 2     # secondes entre chaque snapshot

print('✅ Configuration chargée.')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Configuration chargée.


In [3]:
import os
import sys

def upload_modele_keras():
    """
    Gère l'upload du modèle CNN selon l'environnement :
      - Google Colab  → widget files.upload()
      - Jupyter local → saisie manuelle du chemin
    Met à jour la variable globale CNN_MODEL_PATH.
    """
    global CNN_MODEL_PATH

    # ── Détection de l'environnement ──────────────────────────────────────────
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    if IN_COLAB:
        # ── Google Colab : widget d'upload natif ──────────────────────────────
        from google.colab import files
        print('📤 Sélectionnez votre fichier .keras ...')
        uploaded = files.upload()

        if not uploaded:
            print('⚠️  Aucun fichier uploadé.')
            return

        # Prendre le premier fichier .keras uploadé
        for nom_fichier in uploaded.keys():
            if nom_fichier.endswith('.keras') or nom_fichier.endswith('.h5'):
                CNN_MODEL_PATH = nom_fichier
                taille_mb = len(uploaded[nom_fichier]) / (1024 * 1024)
                print(f'✅ Modèle uploadé : {CNN_MODEL_PATH} ({taille_mb:.1f} Mo)')
                return

        print('⚠️  Aucun fichier .keras / .h5 trouvé parmi les fichiers uploadés.')

    else:
        # ── Jupyter local : saisie ou détection automatique ───────────────────
        print('💻 Environnement local détecté.')
        print()

        # Chercher automatiquement un .keras dans le répertoire courant
        keras_trouves = [
            f for f in os.listdir('.')
            if f.endswith('.keras') or f.endswith('.h5')
        ]

        if keras_trouves:
            print('🔍 Fichiers .keras / .h5 détectés dans le répertoire courant :')
            for i, f in enumerate(keras_trouves):
                taille = os.path.getsize(f) / (1024 * 1024)
                print(f'   [{i}] {f}  ({taille:.1f} Mo)')
            print()
            print(f'→ Utilisation automatique de : {keras_trouves[0]}')
            CNN_MODEL_PATH = keras_trouves[0]
        else:
            print(f'📂 Répertoire courant : {os.getcwd()}')
            print('Aucun fichier .keras trouvé automatiquement.')
            chemin = input('Entrez le chemin complet du fichier .keras : ').strip()
            if os.path.exists(chemin):
                CNN_MODEL_PATH = chemin
                taille_mb = os.path.getsize(chemin) / (1024 * 1024)
                print(f'✅ Chemin défini : {CNN_MODEL_PATH} ({taille_mb:.1f} Mo)')
            else:
                print(f'❌ Fichier introuvable : {chemin}')
                print(f'   CNN_MODEL_PATH reste : {CNN_MODEL_PATH}')

# Lancer l'upload
upload_modele_keras()

📤 Sélectionnez votre fichier .keras ...


Saving mon_modele_v2.keras to mon_modele_v2.keras
✅ Modèle uploadé : mon_modele_v2.keras (24.3 Mo)


In [4]:
if not os.path.exists(CNN_MODEL_PATH):
    raise FileNotFoundError(
        f'❌ Modèle CNN introuvable : {CNN_MODEL_PATH}\n'
        f'   → Relancez la Cellule 3 pour uploader votre fichier .keras.'
    )

print(f'⏳ Chargement du CNN depuis : {CNN_MODEL_PATH}')
_cnn: tf.keras.Model = tf.keras.models.load_model(CNN_MODEL_PATH)
print('✅ CNN chargé.')

print(f'⏳ Chargement de YOLOv8 ({YOLO_MODEL_PATH}) ...')
_yolo: YOLO = YOLO(YOLO_MODEL_PATH)
print('✅ YOLO chargé.')

print()
print('─' * 50)
print(f'  Modèle CNN  : {CNN_MODEL_PATH}')
print(f'  Modèle YOLO : {YOLO_MODEL_PATH}')
print(f'  Classes CNN : {CLASSES}')
print('─' * 50)

⏳ Chargement du CNN depuis : mon_modele_v2.keras
✅ CNN chargé.
⏳ Chargement de YOLOv8 (yolov8n.pt) ...
✅ YOLO chargé.

──────────────────────────────────────────────────
  Modèle CNN  : mon_modele_v2.keras
  Modèle YOLO : yolov8n.pt
  Classes CNN : ['bus', 'camion', 'moto', 'voiture']
──────────────────────────────────────────────────


In [5]:
def calibration_auto(largeur_frame: int) -> float:
    """
    Estime la résolution spatiale à partir de la largeur de l'image.

    Hypothèse : une voie de circulation occupe ~1/4 de la largeur de frame
    et mesure 3,5 m (norme routière française).

    Args:
        largeur_frame: Largeur de la vidéo en pixels.

    Returns:
        Ratio mètres/pixel (float).

    Exemple:
        >>> calibration_auto(1280)   # vidéo 1280 px de large
        0.010937  # m/px
    """
    largeur_voie_px = largeur_frame / 4.0
    largeur_voie_m  = 3.5
    return largeur_voie_m / largeur_voie_px


# ── Test rapide ───────────────────────────────────────────────────────────────
for w in [640, 1280, 1920]:
    print(f'  Vidéo {w}px → {calibration_auto(w):.5f} m/px')

print('✅ Fonction de calibration définie.')

  Vidéo 640px → 0.02187 m/px
  Vidéo 1280px → 0.01094 m/px
  Vidéo 1920px → 0.00729 m/px
✅ Fonction de calibration définie.


In [6]:
class TrackerVitesse:
    """Calcule la vitesse de chaque véhicule suivi par YOLO."""

    def __init__(self, fps: float, pixel_to_meter: float):
        """
        Args:
            fps:            Fréquence d'images de la vidéo source.
            pixel_to_meter: Facteur de conversion px → m (depuis calibration).
        """
        self.fps            = fps
        self.pixel_to_meter = pixel_to_meter
        # {track_id: [(frame_idx, cx, cy), ...]}
        self._historique: dict = defaultdict(list)
        # {track_id: dernière vitesse calculée en km/h}
        self._vitesses: dict   = {}

    def update(self, track_id: int, cx: int, cy: int, frame_idx: int) -> None:
        """
        Enregistre la position du véhicule et recalcule sa vitesse.

        Args:
            track_id:  Identifiant de tracking YOLO (unique par véhicule).
            cx, cy:    Centre de la boîte englobante en pixels.
            frame_idx: Numéro de la frame courante (incrémenté à chaque frame).
        """
        hist = self._historique[track_id]
        hist.append((frame_idx, cx, cy))

        # Fenêtre glissante : on ne garde que VITESSE_WINDOW points
        if len(hist) > VITESSE_WINDOW:
            hist.pop(0)

        # Pas assez de points pour calculer
        if len(hist) < 2:
            self._vitesses[track_id] = 0.0
            return

        # Calcul sur toute la fenêtre (plus stable qu'une seule frame)
        f0, x0, y0 = hist[0]
        f1, x1, y1 = hist[-1]
        delta_frames = f1 - f0

        if delta_frames == 0:
            return   # division par zéro impossible

        dist_px = float(np.sqrt((x1 - x0) ** 2 + (y1 - y0) ** 2))
        dist_m  = dist_px * self.pixel_to_meter
        temps_s = delta_frames / self.fps
        vitesse = (dist_m / temps_s) * 3.6          # m/s → km/h
        vitesse = min(vitesse, float(VITESSE_MAX))   # plafond anti-aberrations

        self._vitesses[track_id] = round(vitesse, 1)

    def get(self, track_id: int) -> float:
        """Retourne la dernière vitesse calculée pour ce véhicule (0.0 si inconnue)."""
        return self._vitesses.get(track_id, 0.0)

    def reset(self) -> None:
        """Vide l'historique — à appeler avant chaque nouvelle vidéo."""
        self._historique.clear()
        self._vitesses.clear()


print('Classe TrackerVitesse définie.')

✅ Classe TrackerVitesse définie.


In [7]:
class ClassificateurCNN:
    """Encapsule le modèle Keras de classification de véhicules."""

    def __init__(self, model: tf.keras.Model):
        self._model = model

    def predire(self, crop_bgr: np.ndarray) -> Optional[str]:
        """
        Classifie un crop de véhicule extrait de la frame.

        Args:
            crop_bgr: Image BGR (numpy array H×W×3) extraite de la frame.

        Returns:
            Nom de la classe prédite (str) ou None si le crop est trop petit.
        """
        h, w = crop_bgr.shape[:2]
        if h < CNN_MIN_CROP_SIZE or w < CNN_MIN_CROP_SIZE:
            return None   # crop trop petit → classification impossible

        # Prétraitement
        img = cv2.resize(crop_bgr, CNN_INPUT_SIZE).astype('float32') / 255.0
        img = np.expand_dims(img, axis=0)           # (1, 64, 64, 3)

        # Inférence
        pred  = self._model.predict(img, verbose=0) # (1, nb_classes)
        return CLASSES[int(np.argmax(pred))]


print('Classe ClassificateurCNN définie.')

✅ Classe ClassificateurCNN définie.


In [8]:
class Annotateur:
    """Dessine les boîtes, labels et overlays sur une frame OpenCV."""

    @staticmethod
    def vehicule(
        frame: np.ndarray,
        x1: int, y1: int, x2: int, y2: int,
        track_id: int,
        genre: str,
        vitesse_kmh: float,
        alerte: bool,
    ) -> None:
        """Dessine la boîte englobante et le label d'un véhicule."""
        couleur = COULEUR_ALERTE if alerte else COULEURS[genre]
        label   = f'ID{track_id} {genre} {vitesse_kmh:.0f}km/h'
        if alerte:
            label += ' !'
        cv2.rectangle(frame, (x1, y1), (x2, y2), couleur, 2)
        cv2.putText(
            frame, label,
            (x1, max(y1 - 8, 14)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.50, couleur, 2,
        )

    @staticmethod
    def stats_cumulees(frame: np.ndarray, compteur: dict) -> None:
        """Affiche les compteurs cumulés en haut à gauche."""
        y = 30
        for genre, n in compteur.items():
            cv2.putText(
                frame, f'{genre}: {n}', (10, y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 0), 2,
            )
            y += 28

    @staticmethod
    def horodatage(frame: np.ndarray, temps_sec: float, H: int) -> None:
        """Affiche l'horodatage du snapshot en bas à gauche."""
        cv2.putText(
            frame, f't={temps_sec:.0f}s', (10, H - 15),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2,
        )

    @staticmethod
    def calibration(frame: np.ndarray, pixel_to_meter: float, W: int) -> None:
        """Affiche la calibration en cours en haut à droite."""
        cv2.putText(
            frame, f'{pixel_to_meter:.4f} m/px', (W - 200, 22),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1,
        )


print('Classe Annotateur définie.')

✅ Classe Annotateur définie.


In [9]:
@dataclass
class ResultatAnalyse:
    """Regroupe toutes les données produites par le moteur d'analyse."""
    video_path:         str
    compteur:           dict = field(default_factory=dict)
    vitesses_par_genre: dict = field(default_factory=lambda: defaultdict(list))
    alertes:            list = field(default_factory=list)
    snapshots:          list = field(default_factory=list)
    pixel_to_meter:     float = 0.0


def analyser_video_moteur(
    video_path:            str,
    pixel_to_meter_manuel: float,
    seuil_vitesse:         float,
    auto_calibrer:         bool,
    progress_fn=None,
) -> ResultatAnalyse:
    """
    Analyse une vidéo frame par frame.

    Args:
        video_path:            Chemin vers la vidéo source.
        pixel_to_meter_manuel: Calibration manuelle (si auto_calibrer=False).
        seuil_vitesse:         Seuil km/h pour les alertes.
        auto_calibrer:         True → utilise calibration_auto(W).
        progress_fn:           Callback(ratio, desc) pour la progression Gradio.

    Returns:
        ResultatAnalyse avec toutes les données collectées.

    Raises:
        IOError: Si la vidéo ne peut pas être ouverte.
    """
    # ── Ouverture vidéo ───────────────────────────────────────────────────────
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f'Impossible d\'ouvrir : {video_path}')

    FPS          = cap.get(cv2.CAP_PROP_FPS) or 25.0
    W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    FRAMES_SNAP  = max(1, int(FPS * SNAPSHOT_INTERVALLE_SEC))

    # Calibration (une seule fois avant la boucle)
    pixel_to_meter = calibration_auto(W) if auto_calibrer else pixel_to_meter_manuel

    # ── Fichier de sortie ─────────────────────────────────────────────────────
    tmp    = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(tmp.name, fourcc, FPS, (W, H))

    # ── Outils ───────────────────────────────────────────────────────────────
    tracker    = TrackerVitesse(FPS, pixel_to_meter)
    cnn_cls    = ClassificateurCNN(_cnn)
    annot      = Annotateur()

    # ── État interne ──────────────────────────────────────────────────────────
    compteur:           dict = defaultdict(int)
    genres_par_id:      dict = {}
    vitesses_par_genre: dict = defaultdict(list)
    alertes:            list = []
    snapshots:          list = []
    compteur_snap:      dict = defaultdict(int)
    vus_snap:           set  = set()
    frame_idx:          int  = 0

    # ── Boucle principale (frame par frame — supporte les longues vidéos) ─────
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # Progression Gradio
        if progress_fn and TOTAL_FRAMES > 0:
            progress_fn(
                frame_idx / TOTAL_FRAMES,
                desc=f'Frame {frame_idx}/{TOTAL_FRAMES}'
            )

        # ── YOLO : détection + tracking ───────────────────────────────────────
        results    = _yolo.track(
            frame,
            persist=True,
            classes=list(VEHICULE_IDS),
            verbose=False,
            tracker='bytetrack.yaml',
        )
        boxes_data = results[0].boxes

        # Aucune détection dans cette frame
        if boxes_data is None or boxes_data.id is None:
            annot.stats_cumulees(frame, compteur)
            annot.calibration(frame, pixel_to_meter, W)
            writer.write(frame)
            continue

        xyxy      = boxes_data.xyxy.cpu().numpy()
        clss      = boxes_data.cls.cpu().numpy().astype(int)
        track_ids = boxes_data.id.cpu().numpy().astype(int)
        confs     = boxes_data.conf.cpu().numpy()

        # ── Traitement de chaque détection ────────────────────────────────────
        for box, cls_id, track_id, conf in zip(xyxy, clss, track_ids, confs):

            # Filtres : classe véhicule + confiance minimale
            if cls_id not in VEHICULE_IDS or conf < CONF_SEUIL:
                continue

            # Coordonnées clampées (évite les crops hors image)
            x1 = max(0, int(box[0]))
            y1 = max(0, int(box[1]))
            x2 = min(W, int(box[2]))
            y2 = min(H, int(box[3]))
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            # Classification CNN
            genre = cnn_cls.predire(frame[y1:y2, x1:x2])
            if genre is None:
                continue

            # Comptage unique total (par ID global)
            if track_id not in genres_par_id:
                genres_par_id[track_id] = genre
                compteur[genre]         += 1

            # Comptage unique snapshot (par fenêtre de 2s)
            if track_id not in vus_snap:
                vus_snap.add(track_id)
                compteur_snap[genre] += 1

            # Calcul vitesse manuelle
            tracker.update(track_id, cx, cy, frame_idx)
            vitesse_kmh = tracker.get(track_id)

            if vitesse_kmh > 0:
                vitesses_par_genre[genre].append(vitesse_kmh)

            # Alerte vitesse
            alerte = vitesse_kmh > seuil_vitesse
            if alerte:
                alertes.append({
                    'temps':   f'{frame_idx / FPS:.1f}s',
                    'genre':   genre,
                    'id':      int(track_id),
                    'vitesse': vitesse_kmh,
                })

            # Annotation de la boîte
            annot.vehicule(frame, x1, y1, x2, y2, track_id, genre, vitesse_kmh, alerte)

        # ── Snapshot toutes les N secondes ────────────────────────────────────
        if frame_idx % FRAMES_SNAP == 0:
            temps_sec = frame_idx / FPS
            snapshots.append({'temps': temps_sec, 'compteur': dict(compteur_snap)})
            annot.horodatage(frame, temps_sec, H)
            compteur_snap = defaultdict(int)   # réinitialisation fenêtre
            vus_snap      = set()

        # ── Overlays permanents (toutes les frames) ───────────────────────────
        annot.stats_cumulees(frame, compteur)
        annot.calibration(frame, pixel_to_meter, W)

        # Écriture dans le fichier de sortie (TOUTES les frames)
        writer.write(frame)

    cap.release()
    writer.release()

    return ResultatAnalyse(
        video_path         = tmp.name,
        compteur           = dict(compteur),
        vitesses_par_genre = vitesses_par_genre,
        alertes            = alertes,
        snapshots          = snapshots,
        pixel_to_meter     = pixel_to_meter,
    )


print('Moteur d\'analyse défini.')

✅ Moteur d'analyse défini.


In [10]:
def generer_rapport(res: ResultatAnalyse, seuil_vitesse: float) -> str:
    """
    Génère le rapport complet à partir d'un ResultatAnalyse.

    Args:
        res:           Résultat produit par analyser_video_moteur().
        seuil_vitesse: Seuil km/h utilisé lors de l'analyse.

    Returns:
        Rapport formaté (str) prêt à afficher dans Gradio.
    """
    total  = sum(res.compteur.values())
    lignes = [
        f'Calibration     : {res.pixel_to_meter:.4f} m/px',
        f'Total véhicules : {total}',
        '',
    ]

    # ── Tableau par classe ────────────────────────────────────────────────────
    lignes += [
        '┌──────────┬────────┬───────────┬──────────┐',
        '│  Classe  │   Nb   │ Moy km/h  │ Max km/h │',
        '├──────────┼────────┼───────────┼──────────┤',
    ]
    for g in CLASSES:
        n     = res.compteur.get(g, 0)
        vlist = res.vitesses_par_genre.get(g, [])
        if vlist:
            lignes.append(
                f'│ {g:<8} │ {n:^6} │ '
                f'{np.mean(vlist):^9.0f} │ {np.max(vlist):^8.0f} │'
            )
        else:
            lignes.append(f'│ {g:<8} │ {n:^6} │ {"—":^9} │ {"—":^8} │')
    lignes.append('└──────────┴────────┴───────────┴──────────┘')

    # ── Tableau snapshots (évolution toutes les 2s) ───────────────────────────
    lignes += [
        '',
        '┌────────┬──────────┬────────┬──────────┬─────────┐',
        '│ Temps  │ Voiture  │  Moto  │   Bus    │ Camion  │',
        '├────────┼──────────┼────────┼──────────┼─────────┤',
    ]
    for snap in res.snapshots:
        t = f"{snap['temps']:.0f}s"
        v = snap['compteur'].get('voiture', 0)
        m = snap['compteur'].get('moto',    0)
        b = snap['compteur'].get('bus',     0)
        c = snap['compteur'].get('camion',  0)
        lignes.append(f'│ {t:<6} │ {v:^8} │ {m:^6} │ {b:^8} │ {c:^7} │')
    lignes.append('└────────┴──────────┴────────┴──────────┴─────────┘')

    # ── Alertes vitesse ───────────────────────────────────────────────────────
    if res.alertes:
        lignes += [
            '',
            f'⚠️  Alertes vitesse (seuil = {seuil_vitesse} km/h) :',
            '┌─────────┬──────────┬──────┬──────────┐',
            '│  Temps  │  Classe  │  ID  │  km/h    │',
            '├─────────┼──────────┼──────┼──────────┤',
        ]
        for a in res.alertes[:30]:
            lignes.append(
                f"│ {a['temps']:^7} │ {a['genre']:<8} │ "
                f"{a['id']:^4} │ {a['vitesse']:^8.0f} │"
            )
        lignes.append('└─────────┴──────────┴──────┴──────────┘')
        if len(res.alertes) > 30:
            lignes.append(f'  … et {len(res.alertes) - 30} alerte(s) supplémentaire(s)')
    else:
        lignes += ['', '✅ Aucun dépassement de vitesse détecté.']

    return '\n'.join(lignes)


def generer_tableau_gradio(res: ResultatAnalyse) -> list:
    """Retourne les snapshots sous forme de liste pour gr.Dataframe."""
    return [
        [
            f"{s['temps']:.0f}s",
            s['compteur'].get('voiture', 0),
            s['compteur'].get('moto',    0),
            s['compteur'].get('bus',     0),
            s['compteur'].get('camion',  0),
            sum(s['compteur'].values()),
        ]
        for s in res.snapshots
    ]


print('Fonctions de rapport définies.')

✅ Fonctions de rapport définies.


In [11]:
def analyser_video(
    video_path,
    pixel_to_meter_manuel: float,
    seuil_vitesse:         float,
    auto_calibrer:         bool,
    progress=gr.Progress(track_tqdm=True),
):
    """Point d'entrée Gradio : reçoit les inputs, retourne les outputs."""

    if video_path is None:
        erreur = '❌ Aucune vidéo chargée.'
        return None, erreur, 0, 0, 0, 0, 0, erreur, []

    try:
        res = analyser_video_moteur(
            video_path            = video_path,
            pixel_to_meter_manuel = pixel_to_meter_manuel,
            seuil_vitesse         = seuil_vitesse,
            auto_calibrer         = auto_calibrer,
            progress_fn           = progress,
        )
    except Exception as e:
        msg = f'❌ Erreur lors de l\'analyse : {e}'
        return None, msg, 0, 0, 0, 0, 0, msg, []

    rapport = generer_rapport(res, seuil_vitesse)
    tableau = generer_tableau_gradio(res)
    total   = sum(res.compteur.values())

    return (
        res.video_path,
        rapport,
        res.compteur.get('voiture', 0),
        res.compteur.get('moto',    0),
        res.compteur.get('bus',     0),
        res.compteur.get('camion',  0),
        total,
        rapport,
        tableau,
    )


# ── Construction de l'interface ───────────────────────────────────────────────
with gr.Blocks(title='Détection de véhicules', theme=gr.themes.Base()) as app:

    gr.Markdown("""
    # 🚗 Système de détection et analyse de véhicules
    **Détection YOLO · Classification CNN · Vitesse calculée · Alertes**
    """)

    # ── Ligne 1 : inputs / vidéo annotée ─────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=1):
            video_input = gr.Video(label='📂 Charger une vidéo (toute durée)')

            with gr.Group():
                gr.Markdown('### ⚙️ Calibration')
                auto_calibrer = gr.Checkbox(
                    label='Automatique  (voie ≈ W/4 px = 3,5 m)',
                    value=True,
                )
                pixel_to_meter_manuel = gr.Slider(
                    minimum=0.001, maximum=0.05,
                    value=0.007, step=0.001,
                    label='Manuelle (m/px)',
                    info='Actif uniquement si calibration automatique désactivée',
                )

            with gr.Group():
                gr.Markdown('### 🚨 Alerte vitesse')
                seuil_vitesse = gr.Slider(
                    minimum=10, maximum=200,
                    value=80, step=5,
                    label='Seuil (km/h)',
                )

            btn = gr.Button('▶  Lancer l\'analyse', variant='primary', size='lg')

        with gr.Column(scale=1):
            video_output = gr.Video(label='🎬 Vidéo annotée')

    # ── Ligne 2 : comptages ───────────────────────────────────────────────────
    gr.Markdown('### 🔢 Comptage total détecté')
    with gr.Row():
        nb_voiture = gr.Number(label='🚗 Voitures', value=0, precision=0)
        nb_moto    = gr.Number(label='🏍️ Motos',    value=0, precision=0)
        nb_bus     = gr.Number(label='🚌 Bus',      value=0, precision=0)
        nb_camion  = gr.Number(label='🚛 Camions',  value=0, precision=0)
        nb_total   = gr.Number(label='📊 Total',    value=0, precision=0)

    # ── Ligne 3 : tableau snapshots ───────────────────────────────────────────
    gr.Markdown('### 📈 Évolution du trafic (toutes les 2 secondes)')
    tableau = gr.Dataframe(
        headers  = ['Temps', 'Voitures', 'Motos', 'Bus', 'Camions', 'Total'],
        datatype = ['str', 'number', 'number', 'number', 'number', 'number'],
        label    = 'Trafic par intervalle',
        wrap     = False,
    )

    # ── Ligne 4 : rapport textuel ─────────────────────────────────────────────
    gr.Markdown('### 📋 Rapport complet')
    rapport = gr.Textbox(
        label            = 'Rapport + Alertes',
        lines            = 25,
        show_copy_button = True,
    )

    # ── Connexion bouton → fonction ───────────────────────────────────────────
    btn.click(
        fn      = analyser_video,
        inputs  = [video_input, pixel_to_meter_manuel, seuil_vitesse, auto_calibrer],
        outputs = [
            video_output,
            rapport,
            nb_voiture, nb_moto, nb_bus,
            nb_camion, nb_total,
            rapport,
            tableau,
        ],
    )

print('Interface Gradio construite.')

/tmp/ipykernel_3048/950815435.py:51: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Détection de véhicules', theme=gr.themes.Base()) as app:


✅ Interface Gradio construite.


## Cellule 12 — Lancement

In [12]:
print(f' Lancement avec le modèle : {CNN_MODEL_PATH}')
app.launch(share=True)

🚀 Lancement avec le modèle : mon_modele_v2.keras
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://681cc5cf0bf2b633a8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
